# Notebook 05 — Multi-Dataset Generalization: FD001 / FD002 / FD004

**Goal:** Show that the pipeline generalizes to harder C-MAPSS datasets with
multiple operating conditions and fault modes.

| Dataset | Operating conditions | Fault modes | Training engines |
|---------|---------------------|-------------|------------------|
| FD001   | 1                   | 1           | 100              |
| FD002   | 6                   | 1           | 260              |
| FD003   | 1                   | 2           | 100              |
| FD004   | 6                   | 2           | 249              |

We focus on FD001 (baseline), FD002 (more conditions), and FD004 (hardest).  
For FD002/FD004 we apply **per-operating-condition normalisation** using
k-means clustering on the three operational settings, then z-score-normalise
sensor readings within each cluster.

In [ ]:
import sys; sys.path.insert(0, '..')

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import (
    load_raw, attach_rul_labels, load_test_rul,
    normalize_per_operating_condition, get_last_cycle,
)
from src.features import build_features, get_feature_columns, save_feature_names
from src.evaluate import (
    rmse, mae, nasa_score, evaluate_on_test,
    cost_analysis, run_to_failure_cost,
)
from src.train import train_model, save_model

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

FIGURES_DIR = Path('../reports/figures')
MODELS_DIR  = Path('../models')
REPORTS_DIR = Path('../reports')
PROCESSED_DIR = Path('../data/processed')

print('Imports OK')

---
## 1. Helper: run full pipeline for one dataset

In [ ]:
def run_pipeline(dataset: str, n_clusters: int = 1, lgbm_params: dict = None) -> dict:
    """
    Full pipeline for a single C-MAPSS sub-dataset.

    Steps:
      1. Load raw train / test.
      2. (FD002/FD004) Cluster operating conditions and normalise sensors.
      3. Attach RUL labels (clipped at 125).
      4. Build rolling/slope/drift features.
      5. Train LightGBM with GroupKFold CV.
      6. Evaluate on test set.
      7. Cost analysis.

    Returns dict with metrics and model.
    """
    print(f'\n{'='*60}')
    print(f'  Dataset: {dataset}  (n_clusters={n_clusters})')
    print(f'{'='*60}')

    lgbm_params = lgbm_params or {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 7,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    }

    # ── Load ────────────────────────────────────────────────────────────────
    train_raw = load_raw(dataset, 'train')
    test_raw  = load_raw(dataset, 'test')
    rul_test  = load_test_rul(dataset)

    # ── Per-condition normalisation (FD002/FD004 only) ───────────────────────
    kmeans_model = None
    scalers = {}
    if n_clusters > 1:
        print(f'  Clustering {n_clusters} operating conditions …')
        train_raw, kmeans_model, scalers = normalize_per_operating_condition(
            train_raw, n_clusters=n_clusters, random_state=42
        )
        # Apply same clusters to test
        test_raw_norm = test_raw.copy()
        from src.data import OP_COLS, SENSOR_COLS
        test_raw_norm['_op_cluster'] = kmeans_model.predict(
            test_raw[OP_COLS].values
        )
        for cid, scaler in scalers.items():
            mask = test_raw_norm['_op_cluster'] == cid
            if mask.sum() > 0:
                test_raw_norm.loc[mask, SENSOR_COLS] = scaler.transform(
                    test_raw_norm.loc[mask, SENSOR_COLS]
                )
        test_raw = test_raw_norm.drop(columns=['_op_cluster'])

        # After normalisation, skip EDA-based dropped sensors
        # (create a blank json so build_features doesn't warn)
        import json
        blank_path = PROCESSED_DIR / f'dropped_sensors_{dataset}.json'
        if not blank_path.exists():
            blank_path.write_text(json.dumps([]))

    # ── Attach RUL labels ────────────────────────────────────────────────────
    train_labeled = attach_rul_labels(train_raw, clip=125)

    # ── Feature engineering ──────────────────────────────────────────────────
    print('  Engineering features …')
    train_feat = build_features(
        train_labeled, dataset=dataset, processed_dir=PROCESSED_DIR,
        include_op_cols=(n_clusters > 1),
    )
    test_feat = build_features(
        test_raw, dataset=dataset, processed_dir=PROCESSED_DIR,
        include_op_cols=(n_clusters > 1),
    )

    feature_cols = get_feature_columns(train_feat)
    save_feature_names(feature_cols, dataset, PROCESSED_DIR)

    X = train_feat[feature_cols].values
    y = train_feat['rul'].values.astype(float)
    groups = train_feat['unit'].values

    # ── Train LightGBM ───────────────────────────────────────────────────────
    print('  Training LightGBM …')
    result = train_model(
        'lgbm',
        pd.DataFrame(X, columns=feature_cols),
        y, groups,
        params=lgbm_params,
        experiment_name='predictive-maintenance-multidataset',
        run_name=f'lgbm_{dataset}',
        models_dir=MODELS_DIR,
    )
    cv = result['cv_metrics']
    print(f'  CV RMSE: {cv["rmse_mean"]:.1f} ± {cv["rmse_std"]:.1f}')

    # ── Test evaluation ──────────────────────────────────────────────────────
    test_last = get_last_cycle(test_feat)
    test_metrics = evaluate_on_test(
        result['model'], test_last,
        rul_test.values.astype(float),
        feature_cols,
    )
    print(f'  Test RMSE={test_metrics["rmse"]:.1f}  '
          f'MAE={test_metrics["mae"]:.1f}  '
          f'NASA={test_metrics["nasa_score"]:.0f}')

    # ── Cost analysis ────────────────────────────────────────────────────────
    cost_df = cost_analysis(
        rul_test.values.astype(float),
        test_metrics['y_pred'],
    )
    opt_idx = cost_df['total_cost'].idxmin()
    optimal_t    = cost_df.loc[opt_idx, 'threshold']
    optimal_cost = cost_df.loc[opt_idx, 'total_cost']
    rtf_cost = run_to_failure_cost(len(rul_test))
    saving = 100 * (rtf_cost - optimal_cost) / rtf_cost
    print(f'  Optimal T={optimal_t:.0f}  Cost savings vs RTF: {saving:.1f}%')

    return {
        'dataset': dataset,
        'model': result['model'],
        'cv_rmse': cv['rmse_mean'],
        'cv_rmse_std': cv['rmse_std'],
        'test_rmse': test_metrics['rmse'],
        'test_mae':  test_metrics['mae'],
        'test_nasa': test_metrics['nasa_score'],
        'optimal_threshold': optimal_t,
        'saving_vs_rtf_pct': saving,
        'y_pred': test_metrics['y_pred'],
        'y_true': rul_test.values.astype(float),
        'cost_df': cost_df,
    }

print('Pipeline helper defined.')

---
## 2. Run all three datasets

In [ ]:
results = {}

# FD001: single operating condition — no normalisation
results['FD001'] = run_pipeline('FD001', n_clusters=1)

# FD002: 6 operating conditions — cluster-normalise
results['FD002'] = run_pipeline('FD002', n_clusters=6)

# FD004: 6 operating conditions + 2 fault modes — hardest
results['FD004'] = run_pipeline('FD004', n_clusters=6)

---
## 3. Operating condition visualisation — FD002

In [ ]:
from src.data import load_raw, OP_COLS
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

fd002_raw = load_raw('FD002', 'train')

# Cluster the 6 operating conditions
km = KMeans(n_clusters=6, random_state=42, n_init=10)
labels = km.fit_predict(fd002_raw[OP_COLS].values)

# Project to 2D for visualisation
pca = PCA(n_components=2)
op_2d = pca.fit_transform(fd002_raw[OP_COLS].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: PCA of operating conditions coloured by cluster
for cl in range(6):
    mask = labels == cl
    axes[0].scatter(op_2d[mask, 0], op_2d[mask, 1], s=5, alpha=0.4,
                    label=f'Cluster {cl}')
axes[0].set_xlabel('PCA dim 1')
axes[0].set_ylabel('PCA dim 2')
axes[0].set_title('FD002 — 6 operating condition clusters (PCA)')
axes[0].legend(markerscale=3, fontsize=8)

# Distribution of sensors per cluster (s2 as example)
fd002_raw['cluster'] = labels
for cl in range(6):
    axes[1].hist(
        fd002_raw.loc[fd002_raw['cluster'] == cl, 's2'],
        bins=30, alpha=0.5, label=f'Cluster {cl}'
    )
axes[1].set_xlabel('s2 raw value')
axes[1].set_ylabel('Count')
axes[1].set_title('s2 distribution per cluster — motivates per-cluster normalisation')
axes[1].legend(fontsize=8)

fig.suptitle(
    'Takeaway: 6 operating conditions create distinct sensor distributions.\n'
    'Normalising per cluster makes the degradation signal comparable across conditions.',
    fontsize=10, y=1.01
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_multidataset_operating_conditions.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_multidataset_operating_conditions.png')

---
## 4. Results comparison table

In [ ]:
rows = []
for ds, r in results.items():
    rows.append({
        'Dataset': ds,
        'Operating conditions': {'FD001': 1, 'FD002': 6, 'FD004': 6}[ds],
        'Fault modes': {'FD001': 1, 'FD002': 1, 'FD004': 2}[ds],
        'Test RMSE': round(r['test_rmse'], 1),
        'Test MAE': round(r['test_mae'], 1),
        'NASA score': int(round(r['test_nasa'])),
        'Optimal T': int(r['optimal_threshold']),
        'Saving vs RTF (%)': round(r['saving_vs_rtf_pct'], 1),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

results_df.to_csv(REPORTS_DIR / 'multidataset_results.csv', index=False)
print('\nSaved to reports/multidataset_results.csv')

---
## 5. Performance degradation visualisation

In [ ]:
datasets = ['FD001', 'FD002', 'FD004']
rmse_vals  = [results[d]['test_rmse']  for d in datasets]
nasa_vals  = [results[d]['test_nasa']  for d in datasets]
saving_vals = [results[d]['saving_vs_rtf_pct'] for d in datasets]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = ['steelblue', 'darkorange', 'tomato']

axes[0].bar(datasets, rmse_vals, color=colors, edgecolor='white')
axes[0].set_ylabel('Test RMSE (cycles)')
axes[0].set_title('RMSE — harder datasets score worse')
for i, v in enumerate(rmse_vals):
    axes[0].text(i, v + 0.5, f'{v:.1f}', ha='center', fontsize=10)

axes[1].bar(datasets, nasa_vals, color=colors, edgecolor='white')
axes[1].set_ylabel('NASA score (lower = better)')
axes[1].set_title('NASA score — asymmetric late-prediction penalty')
for i, v in enumerate(nasa_vals):
    axes[1].text(i, v + max(nasa_vals)*0.01, f'{v:.0f}', ha='center', fontsize=10)

axes[2].bar(datasets, saving_vals, color=colors, edgecolor='white')
axes[2].set_ylabel('Cost saving vs run-to-failure (%)')
axes[2].set_title('Business value — still significant on harder datasets')
axes[2].axhline(0, color='black', lw=0.8)
for i, v in enumerate(saving_vals):
    axes[2].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

fig.suptitle(
    'Performance degrades from FD001 → FD004 (more operating conditions + fault modes).\n'
    'Per-condition normalisation partially recovers performance on FD002/FD004.',
    fontsize=10, y=1.02
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_multidataset_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_multidataset_comparison.png')

---
## 6. Why does performance degrade from FD001 → FD004?

**FD002** has six distinct operating conditions (altitude/speed/throttle
combinations). Without per-condition normalisation, a sensor reading of
`s2=540` means something very different in low-altitude vs high-altitude
cruise. The model would confuse operating-condition differences with
degradation. Cluster-based z-score normalisation removes this confound,
but residual noise from imperfect clustering still hurts RMSE.

**FD004** compounds this with two fault modes. The degradation trajectory
looks fundamentally different depending on which component is wearing out,
and the model has no label indicating fault mode. A production system would
identify fault mode via anomaly detection or domain knowledge and train
separate models per mode.

**Interventions that would help:**
- Fault-mode clustering (unsupervised) before RUL modelling.
- Per-engine normalisation (but this requires sufficient run-up data).
- LSTM/Transformer sequence models that are less sensitive to operating-
  condition shifts — they are more useful on FD002/FD004 than on FD001.
- Richer operating-condition features (sine/cosine embeddings of the six
  discrete conditions).